In [ ]:
%pip install -U numpy matplotlib imageio tifffile pillow ipython scipy scikit-image ipython

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
import tifffile as tiff
from PIL import Image
from IPython.display import display
from scipy.ndimage import gaussian_filter
from scipy import ndimage as ndi
from skimage.morphology import remove_small_objects, disk, opening, closing
from skimage.segmentation import clear_border
from typing import List, Tuple, Dict, Optional, Iterable

from scipy.special import j0
from scipy.optimize import least_squares
try:
    from scipy.optimize import curve_fit
except Exception:  # pragma: no cover
    curve_fit = None


In [ ]:
def find_project_root(
    start: str | Path = ".",
    *,
    required_subpath: str = "pictures",
    max_up: int = 10,
    create_if_missing: bool = True,
) -> Path:
    """
    Walk upwards to find a directory containing `required_subpath`.
    If not found and create_if_missing=True, create it under `start` and return `start`.
    """
    start_path = Path(start).resolve()
    p = start_path
    for _ in range(max_up + 1):
        if (p / required_subpath).exists():
            return p
        if p.parent == p:
            break
        p = p.parent

    if create_if_missing:
        (start_path / required_subpath).mkdir(parents=True, exist_ok=True)
        return start_path

    raise FileNotFoundError(
        f"Cannot find project root from {start_path} (missing {required_subpath}). "
        f"Set root=Path('...') to your repo root."
    )
root = find_project_root(".")
pictures_dir = root / "pictures"
pictures_dir.mkdir(parents=True, exist_ok=True)

def _to_float01(img: np.ndarray) -> np.ndarray: # 将图片数据转换为0-1范围的浮点数
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    if img.ndim == 3 and img.shape[-1] == 4:
        img = img[..., :3]
    if img.ndim != 3 or img.shape[-1] != 3:
        raise ValueError(f"Expect RGB image, got {img.shape}")

    if np.issubdtype(img.dtype, np.floating): # 如果已经是浮点数类型，检查是否需要缩放
        x = img.astype(np.float32, copy=False)
        if np.nanmax(x) > 1.0:
            x = x / 255.0
    else:
        x = img.astype(np.float32) / 255.0
    return np.clip(x, 0.0, 1.0)




def slider_c(val: float) -> float:
    """val in [-100,100] -> c in [-1,1]"""
    return float(np.clip(val, -100, 100) / 100.0)

def slider_factor(val: float) -> float:
    """val in [-100,100] -> factor in [0,2]"""
    c = slider_c(val)
    return float(np.clip(1.0 + c, 0.0, 2.0))
def rgb_to_luma(img01: np.ndarray) -> np.ndarray:
    return (0.2126 * img01[..., 0] +
            0.7152 * img01[..., 1] +
            0.0722 * img01[..., 2])

def adjust_clarity_luma(img01: np.ndarray, clarity: float, sigma: float = 2.0) -> np.ndarray:
    c = slider_c(clarity)  # [-1,1]
    L = rgb_to_luma(img01)                      # (H,W)
    L_blur = gaussian_filter(L, sigma=sigma)    # (H,W)
    L2 = np.clip(L + c * (L - L_blur), 0.0, 1.0)
    eps = 1e-6
    ratio = (L2 + eps) / (L + eps)              # (H,W)
    out = img01 * ratio[..., None]
    return np.clip(out, 0.0, 1.0)

def adjust_brightness(img01: np.ndarray, brightness: float) -> np.ndarray:
    f = slider_factor(brightness)
    return np.clip(img01 * f, 0.0, 1.0)
def adjust_highlights(img01: np.ndarray, highlights: float) -> np.ndarray:
    f = slider_factor(highlights)
    out = img01 + (1.0 - img01) * (f - 1.0)
    return np.clip(out, 0.0, 1.0)

def adjust_contrast(img01: np.ndarray, contrast: float) -> np.ndarray:
    f = slider_factor(contrast)
    return np.clip((img01 - 0.5) * f + 0.5, 0.0, 1.0)

def adjust_saturation(img01: np.ndarray, saturation: float) -> np.ndarray:
    f = slider_factor(saturation)
    luma = (0.2126 * img01[..., 0] + 0.7152 * img01[..., 1] + 0.0722 * img01[..., 2])[..., None]
    out = luma + (img01 - luma) * f
    return np.clip(out, 0.0, 1.0)

def adjust_warmth(img01: np.ndarray, warmth: float) -> np.ndarray:
    c = slider_c(warmth)
    r_factor = float(np.clip(1.0 + c, 0.0, 2.0))
    b_factor = float(np.clip(1.0 - c, 0.0, 2.0))

    out = img01.copy()
    out[..., 0] = np.clip(out[..., 0] * r_factor, 0.0, 1.0)  # R
    out[..., 2] = np.clip(out[..., 2] * b_factor, 0.0, 1.0)  # B
    return out

def apply_sliders(img: np.ndarray, clarity=0, brightness=0, contrast=0, saturation=0, highlights=0, warmth=0, debug=False):
    x = _to_float01(img)
    x = adjust_clarity_luma(x, clarity=clarity, sigma=2.0)
    x = adjust_brightness(x, brightness)
    x = adjust_contrast(x, contrast)
    x = adjust_highlights(x, highlights)
    x = adjust_saturation(x, saturation)
    x = adjust_warmth(x, warmth)
    return x

def fill_black_holes(mask: np.ndarray, max_hole_area: int | None = None) -> np.ndarray:
    m = mask.astype(bool)
    filled_all = ndi.binary_fill_holes(m)

    if max_hole_area is None:
        return filled_all

    holes = filled_all & (~m)          # pixels that would be added to foreground
    if not holes.any():
        return m

    lab, n = ndi.label(holes)
    if n == 0:
        return m

    counts = np.bincount(lab.ravel())  # counts[0] is background
    small_ids = np.where(counts <= max_hole_area)[0]
    small_ids = small_ids[small_ids != 0]

    if small_ids.size == 0:
        return m

    small_holes = np.isin(lab, small_ids)
    return m | small_holes

def denoise_mask_conservative(mask_filled: np.ndarray,
                              min_obj_area: int = 12,
                              drop_border: bool = False) -> np.ndarray:
    m = mask_filled.astype(bool)
    
    if drop_border:
        m = clear_border(m)
    m = remove_small_objects(m, min_size=min_obj_area, connectivity=2)
    
    return m.astype(bool)

def find_centroids(
    mask: np.ndarray,
    *,
    labeled: bool = False,
    connectivity: int = 2,
    min_area: int = 1,
    return_labels: bool = False,
) -> List[Tuple[float, float]] | Tuple[List[Tuple[float, float]], np.ndarray]:

    mask = np.asarray(mask)

    if labeled:
        labels_img = mask.astype(np.int32, copy=False)
        if labels_img.ndim != 2:
            raise ValueError("label image must be 2D")
    else:
        if mask.ndim != 2:
            raise ValueError("binary mask must be 2D")
        # treat anything >0 as foreground
        fg = mask > 0
        # connectivity structure
        structure = ndi.generate_binary_structure(rank=2, connectivity=connectivity)
        labels_img, _ = ndi.label(fg, structure=structure)

    if labels_img.max() == 0:
        out = ([], labels_img) if return_labels else []
        return out

    # compute areas per label
    counts = np.bincount(labels_img.ravel())
    # labels are 1..K
    keep = [lab for lab in range(1, len(counts)) if counts[lab] >= int(min_area)]

    if len(keep) == 0:
        out = ([], labels_img) if return_labels else []
        return out

    # center_of_mass returns (cy, cx)
    centroids = ndi.center_of_mass(np.ones_like(labels_img, dtype=np.float32), labels_img, keep)
    centroids = [(float(cy), float(cx)) for (cy, cx) in centroids]

    if return_labels:
        return centroids, labels_img
    return centroids


def centroids_points_image(
    centroids: list[tuple[float, float]],
    shape: tuple[int, int],
    *,
    radius: int = 2,
) -> np.ndarray:

    h, w = shape
    img = np.full((h, w), 255, dtype=np.uint8)

    for cy, cx in centroids:
        y = int(round(cy))
        x = int(round(cx))
        if not (0 <= y < h and 0 <= x < w):
            continue

        if radius <= 0:
            img[y, x] = 0
            continue

        y0, y1 = max(0, y - radius), min(h, y + radius + 1)
        x0, x1 = max(0, x - radius), min(w, x + radius + 1)
        yy, xx = np.ogrid[y0:y1, x0:x1]
        disk = (yy - y) ** 2 + (xx - x) ** 2 <= radius ** 2
        img[y0:y1, x0:x1][disk] = 0

    return img


In [ ]:

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
@dataclass(frozen=True)
class BatchParams:
    # sliders
    clarity: float = 0
    brightness: float = 0
    highlights: float = 0
    contrast: float = 0
    saturation: float = 0
    warmth: float = 0

    # threshold
    T: float = 0.5
    dark_is_fg: bool = True  # True: gray < T; False: gray > T

    # mask cleanup
    max_hole_area: int = 100
    min_obj_area: float = 0.04
    drop_border: bool = True

    # centroids + points
    min_area: int = 20
    point_radius: int = 6

def _iter_images(folder: Path, recursive: bool = False) -> Iterable[Path]:
    it = folder.rglob("*") if recursive else folder.iterdir()
    for p in it:
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p

def process_folder_PhotoCleaner(
    in_dir: str | Path,
    out_bw_dir: str | Path,
    out_points_dir: str | Path,
    *,
    params: BatchParams = BatchParams(),
    recursive: bool = False,
    display_requested: bool = False,
) -> None:

    in_dir = Path(in_dir)
    out_bw_dir = Path(out_bw_dir)
    out_points_dir = Path(out_points_dir)


    if not in_dir.is_dir():
        print(f"Input directory does not exist: {in_dir}")
        return
    out_bw_dir.mkdir(parents=True, exist_ok=True)
    out_points_dir.mkdir(parents=True, exist_ok=True)
    imgs = list(_iter_images(in_dir, recursive=recursive))
    if not imgs:
        print(f"No images found in: {in_dir}")
        return

    for p in imgs:
        try:
            img = np.asarray(Image.open(p).convert("RGB"))

            # 1) sliders (returns float01 RGB)
            out_change_threshold = apply_sliders(
                img,
                clarity=params.clarity,
                brightness=params.brightness,
                highlights=params.highlights,
                contrast=params.contrast,
                saturation=params.saturation,
                warmth=params.warmth,
            )

            # 2) gray + threshold
            gray = (
                0.2126 * out_change_threshold[..., 0]
                + 0.7152 * out_change_threshold[..., 1]
                + 0.0722 * out_change_threshold[..., 2]
            )
            mask = gray < params.T if params.dark_is_fg else gray > params.T  # bool

            # 3) fill holes + denoise (bool in/bool out)
            mask = fill_black_holes(mask, max_hole_area=params.max_hole_area)
            mask = denoise_mask_conservative(
                mask,
                min_obj_area=params.min_obj_area,
                drop_border=params.drop_border,
            )

            # 4) centroids on bool mask
            centroids = find_centroids(mask, min_area=params.min_area)


            bw = (~mask).astype(np.uint8) * 255  # white bg, black fg
            points = centroids_points_image(centroids, mask.shape, radius=params.point_radius)

            Image.fromarray(bw, mode="L").save(out_bw_dir / f"{p.stem}_bcst.test1.png")
            Image.fromarray(points, mode="L").save(out_points_dir / f"{p.stem}_centroids.test1.png")

            print(f"[OK] {p.name}  centroids={len(centroids)}")
            if display_requested:
                display(Image.fromarray(img))          # 已经是 RGB
                display(Image.fromarray(bw, mode="L"))
                display(Image.fromarray(points, mode="L"))

        except Exception as e:
            print(f"[FAIL] {p.name}: {e}")

In [ ]:
#auto run all images in the folder with the same parameters:
root = find_project_root(".") # 你项目根目录（ipynb所在目录通常就是）
in_dir = root / "pictures" / "pictures_in"
if not in_dir.is_dir():
    in_dir.mkdir(parents=True, exist_ok=True)
    print(f"you don't have in_dir, I created: {in_dir} for you, put your images there and re-run this cell.")
    

params = BatchParams(
    clarity=30,
    brightness=70,
    highlights=-100,
    contrast=99,
    saturation=-99,
    warmth=0,
    T=0.5,
    dark_is_fg=True,
    max_hole_area=100,
    min_obj_area=0.2,
    drop_border=False,
    min_area=20,
    point_radius=6,
)

process_folder_PhotoCleaner(
    root / "pictures" / "pictures_in",
    root / "pictures" / "pic_blacknwhite",
    root / "pictures" / "pic_centroids",
    params=params,
    recursive=False,
    display_requested=True,
)

In [ ]:
def points_from_binary_png_cc(path: str, *, white_is_cell: bool = True,
                              min_area: int = 10, connectivity: int = 2):
    """
    Extract cell centroids from a binary PNG using connected components.

    Parameters
    ----------
    path : str
        Path to PNG.
    white_is_cell : bool
        If True, pixels > 0 are treated as foreground (cells).
    min_area : int
        Remove small components (noise).
    connectivity : int
        1=4-neighborhood, 2=8-neighborhood in 2D.

    Returns
    -------
    points_xy : (N,2) float array
        Each row is (x, y) in pixel coordinates.
    mask : 2D bool array
        Foreground mask used.
    """
    img = Image.open(path).convert("L")
    arr = np.asarray(img, dtype=np.uint8)

    mask = arr > 0 if white_is_cell else arr == 0
    mask = mask.astype(bool)

    struct = ndi.generate_binary_structure(2, connectivity)
    labeled, n = ndi.label(mask, structure=struct)
    if n == 0:
        return np.zeros((0, 2), dtype=float), mask

    # filter by size
    counts = np.bincount(labeled.ravel())
    keep = np.zeros_like(counts, dtype=bool)
    keep[1:] = counts[1:] >= int(min_area)

    labeled_kept = labeled.copy()
    labeled_kept[~keep[labeled_kept]] = 0
    labeled_kept, n2 = ndi.label(labeled_kept > 0, structure=struct)
    if n2 == 0:
        return np.zeros((0, 2), dtype=float), mask

    centroids_yx = ndi.center_of_mass(labeled_kept > 0, labeled_kept, index=np.arange(1, n2 + 1))
    centroids_yx = np.array(centroids_yx, dtype=float)  # (N,2) in (y,x)
    points_xy = centroids_yx[:, ::-1]  # to (x,y)
    return points_xy, mask

def c2p_2d(points_xy: np.ndarray, r_max: float, dr: float, area: float | None = None):
    """
    Estimate 2D isotropic radial 2-point correlation function:
      g(r) = observed_pairs_in_ring / expected_pairs_in_ring(Poisson)
      c2p(r) = g(r) - 1

    Parameters
    ----------
    points_xy : (N,2) array
    r_max     : max radius (same unit as coordinates)
    dr        : radial bin width
    area      : ROI area; if None uses AABB area (pragmatic)

    Returns
    -------
    r   : bin centers
    c2p : correlation (g-1)
    g   : pair correlation
    n   : number density (#/area)
    """
    pts = np.asarray(points_xy, dtype=float)
    if pts.ndim != 2 or pts.shape[1] != 2:
        raise ValueError("points_xy must be (N,2)")
    pts = pts[np.all(np.isfinite(pts), axis=1)]
    n_pts = pts.shape[0]
    if n_pts < 5:
        raise ValueError("Too few points")

    if area is None:
        mins = pts.min(axis=0)
        maxs = pts.max(axis=0)
        spans = np.maximum(maxs - mins, 1e-12)
        area = float(spans[0] * spans[1])

    density = float(n_pts / area)

    # pair distances (upper triangle)
    diffs = pts[:, None, :] - pts[None, :, :]
    d2 = np.sum(diffs * diffs, axis=-1)
    iu = np.triu_indices(n_pts, k=1)
    rij = np.sqrt(d2[iu])

    edges = np.arange(0.0, r_max + dr, dr, dtype=float)
    counts, _ = np.histogram(rij, bins=edges)

    r = 0.5 * (edges[:-1] + edges[1:])
    shell = edges[1:] - edges[:-1]
    ring_area = 2.0 * np.pi * r * shell

    expected = 0.5 * n_pts * density * ring_area
    expected = np.maximum(expected, 1e-30)

    g = counts / expected
    c2p = g - 1.0
    return r, c2p, g, density

def psd_from_c2p_2d(r: np.ndarray, c2p: np.ndarray, density: float,
                   k_min: float, k_max: float, num_k: int = 120,
                   add_shot_noise: bool = True):
    """
    2D isotropic PSD from c2p(r):
      P(k) = 2π ∫ c(r) J0(kr) r dr  + shot_noise

    shot_noise is approximated as 1/density (pragmatic baseline).
    If you have absolute normalization / ROI, replace accordingly.

    Returns
    -------
    k : log-spaced wave numbers
    P : PSD values (positive)
    """
    r = np.asarray(r, dtype=float)
    c = np.asarray(c2p, dtype=float)
    if r.shape != c.shape:
        raise ValueError("r and c2p must have same shape")

    k = np.logspace(np.log10(k_min), np.log10(k_max), num=num_k).astype(float)

    dr = np.gradient(r)
    weights = (2.0 * np.pi) * (c * r * dr)  # (Nr,)

    kr = np.outer(k, r)                     # (Nk,Nr)
    P = weights @ j0(kr).T                  # (Nk,)
    P = P.astype(float)

    shot = (1.0 / max(density, 1e-30)) if add_shot_noise else 0.0
    P = np.maximum(P + shot, 1e-300)
    return k, P

#parameter fitting test function 
rd_psd_model_2d = lambda k, D, gamma, dt, P0: P0 / (1.0 + (dt * D * k**2) / gamma)  
def fit_rd_psd_2d(k: np.ndarray, P: np.ndarray, dt: float,
                  fit_P0: bool = True, robust: bool = True,
                  k_quantile_range: Tuple[float, float] = (0.1, 0.9)):
    """
    Fit (D, gamma, P0) to PSD using log10 residuals for numerical stability.

    k_quantile_range: fit only mid-range k (avoid low-k windowing + high-k noise).
    """
    k = np.asarray(k, float)
    P = np.asarray(P, float)

    good = np.isfinite(k) & np.isfinite(P) & (k > 0) & (P > 0)
    k = k[good]
    P = np.maximum(P[good], 1e-300)
    if k.size < 10:
        raise ValueError("Not enough valid points for fitting")

    # choose fitting window in k
    qlo, qhi = k_quantile_range
    k_lo = np.quantile(k, qlo)
    k_hi = np.quantile(k, qhi)
    sel = (k >= k_lo) & (k <= k_hi)
    if np.count_nonzero(sel) >= 10:
        k = k[sel]
        P = P[sel]

    # initial guesses
    P0_init = float(np.median(P[:max(3, len(P)//8)]))
    target = P0_init / 4.0
    idx = int(np.argmin(np.abs(P - target)))
    k_c = float(k[idx])
    ratio_init = 1.0 / (max(dt, 1e-30) * max(k_c, 1e-30) ** 2)  # ~ D/gamma
    gamma_init = 1.0
    D_init = float(np.clip(ratio_init * gamma_init, 1e-12, 1e12))

    # log-parameterization for stability
    if fit_P0:
        x0 = np.array([np.log(D_init), np.log(gamma_init), np.log(np.clip(P0_init, 1e-300, 1e300))], float)
        lo = np.array([np.log(1e-12), np.log(1e-12), np.log(1e-300)], float)
        hi = np.array([np.log(1e12),  np.log(1e12),  np.log(1e300)], float)
    else:
        x0 = np.array([np.log(D_init), np.log(gamma_init)], float)
        lo = np.array([np.log(1e-12), np.log(1e-12)], float)
        hi = np.array([np.log(1e12),  np.log(1e12)], float)

    logP = np.log10(P)

    def resid(x):
        if fit_P0:
            logD, logG, logP0 = map(float, x)
            D, gamma, P0 = np.exp(logD), np.exp(logG), np.exp(logP0)
        else:
            logD, logG = map(float, x)
            D, gamma, P0 = np.exp(logD), np.exp(logG), P0_init

        pred = rd_psd_model_2d(k, D=D, gamma=gamma, dt=dt, P0=P0)
        return np.log10(pred) - logP

    res = least_squares(
        resid,
        x0=x0,
        bounds=(lo, hi),
        loss="soft_l1" if robust else "linear",
        f_scale=0.25,
        max_nfev=20000,
    )

    if fit_P0:
        D_hat, g_hat, P0_hat = np.exp(res.x[0]), np.exp(res.x[1]), np.exp(res.x[2])
    else:
        D_hat, g_hat, P0_hat = np.exp(res.x[0]), np.exp(res.x[1]), P0_init

    pred = rd_psd_model_2d(k, D=float(D_hat), gamma=float(g_hat), dt=float(dt), P0=float(P0_hat))
    rmse_log10 = float(np.sqrt(np.mean((np.log10(pred) - np.log10(P)) ** 2)))

    return {
        "D": float(D_hat),
        "gamma": float(g_hat),
        "P0": float(P0_hat),
        "dt": float(dt),
        "rmse_log10": rmse_log10,
        "success": bool(res.success),
        "message": str(res.message),
        "nfev": int(res.nfev),
    }
#final fucniton 

def process_folder_psd(
    in_dir: str | Path,
    *,
    r_max: float,
    dr: float,
    k_min: float,
    k_max: float,
    num_k: int = 120,
    fit_P0: bool = True,
    fit_dt: bool = False,
) -> None:
    in_dir = Path(in_dir)

    if not in_dir.is_dir():
        print(f"Input directory does not exist: {in_dir}")
        return

    for p in _iter_images(in_dir):
        try:
            points_xy, mask = points_from_binary_png_cc(p, white_is_cell=False, min_area=10)
            if len(points_xy) < 5:
                print(f"[SKIP] {p.name}: too few points")
                continue

            r, c2p, g, density = c2p_2d(points_xy, r_max=r_max, dr=dr)
            k, P = psd_from_c2p_2d(r, c2p, density, k_min=k_min, k_max=k_max, num_k=num_k)
            fit_res = fit_psd_params(k, P, dt=1.0, fit_P0=True, robust=True)

            Data = r, c2p, g, k, P, fit_res
            return Data    
        except Exception as e:
            print( f"[FAIL] {p.name}: {e}")
            continue
    return None



In [ ]:
#plot campare c2p and psd for multiple sources:
def plot_c2p_compare(
    Datas: dict,
    *,
    title: str = "c2p(r) comparison",
    use_g: bool = False,
    alpha: float = 0.9,
):
    """
    Plot c2p(r) (or g(r)) for multiple sources on one figure.

    Datas: dict[name -> (r, c2p, g, k, P, fit_res)]
    """
    plt.figure()

    for name, data in Datas.items():
        r, c2p, g, *_ = data
        r = np.asarray(r)
        y = np.asarray(g if use_g else c2p)
        plt.plot(r, y, label=name, alpha=alpha)

    plt.axhline(1.0 if use_g else 0.0, linewidth=1)
    plt.xlabel("r")
    plt.ylabel("g(r)" if use_g else "c2p(r)=g(r)-1")
    plt.title(title)
    plt.legend()
    plt.show()


def plot_psd_compare(Datas: dict, *, title: str = "PSD compare", alpha: float = 0.9):
    """
    Datas: {name: (r, c2p, g, k, P, fit_res)}
    """
    plt.figure()
    for name, data in Datas.items():
        k = np.asarray(data[3], float)
        P = np.asarray(data[4], float)
        plt.loglog(k, P, label=name, alpha=alpha)

    plt.xlabel("k")
    plt.ylabel("P(k)")
    plt.title(title)
    plt.legend()
    plt.show()

def plot_residuals_compare(Datas: dict, model_fn, *, title="Residuals compare"):
    """
    Datas: {name: (r, c2p, g, k, P, fit_res)}
    model_fn: function(k, D=..., gamma=..., dt=..., P0=...) -> P_model
    """
    plt.figure()
    for name, data in Datas.items():
        k = np.asarray(data[3], float)
        P = np.maximum(np.asarray(data[4], float), 1e-300)
        fit_res = data[5]

        Pm = model_fn(k, D=fit_res["D"], gamma=fit_res["gamma"], dt=fit_res["dt"], P0=fit_res["P0"])
        Pm = np.maximum(np.asarray(Pm, float), 1e-300)

        resid = np.log10(Pm) - np.log10(P)
        plt.plot(k, resid, label=name)

    plt.xscale("log")
    plt.axhline(0.0, linewidth=1)
    plt.xlabel("k")
    plt.ylabel("log10(P_fit) - log10(P_data)")
    plt.title(title)
    plt.legend()
    plt.show()

def plot_fit_compare(Datas: dict, model_fn, *, title="Fit results"):
    """
    Plot fit results for multiple sources.
    """
    plt.figure()
    for name, data in Datas.items():
        k = np.asarray(data[3], float)
        P = np.maximum(np.asarray(data[4], float), 1e-300)
        fit_res = data[5]

        Pm = model_fn(k, D=fit_res["D"], gamma=fit_res["gamma"], dt=fit_res["dt"], P0=fit_res["P0"])
        Pm = np.maximum(np.asarray(Pm, float), 1e-300)

        plt.loglog(k, P, label=f"{name} (data)")
        plt.loglog(k, Pm, label=f"{name} (model)", linestyle="--")

    plt.xlabel("k")
    plt.ylabel("P(k)")
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
root = find_project_root(".")
in_dirs = [
    ("bw", root / "pictures" / "pic_blacknwhite"),
    ("centroids", root / "pictures" / "pic_centroids"),
]

Datas = {}  # ✅ 放在循环外

for name, in_dir in in_dirs:
    data = process_folder_psd(
        in_dir,
        r_max=100.0,
        dr=1.0,
        k_min=0.01,
        k_max=10.0,
        num_k=120,
    )
    if data is not None:
        Datas[name] = data

plot_c2p_compare(Datas, title="c2p compare")
plot_psd_compare(Datas, title="PSD compare")
plot_residuals_compare(Datas, model_fn=rd_psd_model_2d, title="PSD fit residuals") 
plot_fit_compare(Datas, model_fn=rd_psd_model_2d, title="PSD fit compare")


